# ML Evaluation Metrics

How to measure whether a model is actually good.

---

## Table of Contents
1. [Regression Metrics](#1-regression-metrics)
2. [Confusion Matrix](#2-confusion-matrix)
3. [Accuracy, Precision, Recall, F1](#3-accuracy-precision-recall-f1)
4. [Classification Threshold Effects](#4-classification-threshold-effects)
5. [ROC Curve & AUC](#5-roc-curve--auc)
6. [Log-loss & Cross-entropy](#6-log-loss--cross-entropy)

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from sklearn.metrics import roc_curve, auc, confusion_matrix, precision_recall_curve, log_loss
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import train_test_split
import os

os.makedirs('img', exist_ok=True)

plt.rcParams.update({
    'figure.facecolor': '#0f0f0f', 'axes.facecolor': '#1a1a1a',
    'axes.edgecolor': '#444', 'axes.labelcolor': '#ccc',
    'xtick.color': '#888', 'ytick.color': '#888', 'text.color': '#eee',
    'grid.color': '#2a2a2a', 'grid.linewidth': 0.8,
    'font.family': 'monospace', 'axes.titlesize': 12, 'axes.labelsize': 11,
})
ACCENT = '#00e5ff'
ORANGE = '#ff6d00'
GREEN  = '#69ff47'
RED    = '#ff4d6d'
PURPLE = '#c77dff'
YELLOW = '#ffd60a'

---
## 1. Regression Metrics

**MAE** — Mean Absolute Error:
$$\text{MAE} = \frac{1}{n}\sum_{i=1}^n |y_i - \hat{y}_i|$$
Same unit as $y$. Robust to outliers. Minimising MAE predicts the **median**.

**MSE** — Mean Squared Error:
$$\text{MSE} = \frac{1}{n}\sum_{i=1}^n (y_i - \hat{y}_i)^2$$
Penalises large errors quadratically. Minimising MSE predicts the **mean**.

**RMSE** = $\sqrt{\text{MSE}}$ — same unit as $y$, still sensitive to outliers.

**R²** — Coefficient of Determination:
$$R^2 = 1 - \frac{\sum(y_i - \hat{y}_i)^2}{\sum(y_i - \bar{y})^2}$$
$R^2=1$ perfect, $R^2=0$ no better than the mean, $R^2<0$ worse than the mean.

In [ ]:
np.random.seed(0)
x_reg = np.linspace(0, 10, 50)
y_true = 2*x_reg + 1 + np.random.normal(0, 2, 50)
y_true[25] += 18

coef_all    = np.polyfit(x_reg, y_true, 1)
mask        = np.ones(50, dtype=bool); mask[25] = False
coef_no_out = np.polyfit(x_reg[mask], y_true[mask], 1)
y_pred_all    = np.polyval(coef_all, x_reg)
y_pred_no_out = np.polyval(coef_no_out, x_reg)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Regression Metrics', fontsize=13)

ax = axes[0]
ax.scatter(x_reg, y_true, color=ACCENT, s=30, zorder=5, label='data')
ax.scatter(x_reg[25], y_true[25], color=RED, s=120, zorder=6, marker='*', label='outlier')
ax.plot(x_reg, y_pred_all,    color=RED,   lw=2, linestyle='--', label='fit with outlier')
ax.plot(x_reg, y_pred_no_out, color=GREEN, lw=2, label='fit without outlier')
ax.set_xlabel('$x$'); ax.set_ylabel('$y$')
ax.set_title('One outlier can distort the entire fit')
ax.legend(fontsize=9); ax.grid(True)

ax = axes[1]
outer = np.linspace(0, 30, 100)
maes  = [np.mean(np.abs(np.array([0.0]*25 + [m] + [0.0]*24))) for m in outer]
rmses = [np.sqrt(np.mean(np.array([0.0]*25 + [m] + [0.0]*24)**2)) for m in outer]
ax.plot(outer, maes,  color=GREEN,  lw=2.5, label='MAE — grows linearly')
ax.plot(outer, rmses, color=ORANGE, lw=2.5, label='RMSE — grows faster')
ax.set_xlabel('Outlier magnitude'); ax.set_ylabel('Error')
ax.set_title('MAE vs RMSE sensitivity to outliers')
ax.legend(fontsize=10); ax.grid(True)

ax = axes[2]
labels_r2 = ['Perfect fit', 'Good model', 'Mean baseline', 'Bad model']
r2_vals   = [1.0, 0.82, 0.0, -0.35]
colors_r2 = [ACCENT, GREEN, YELLOW, RED]
bars = ax.barh(labels_r2, r2_vals, color=colors_r2, alpha=0.8, height=0.5)
ax.axvline(0, color='white', lw=1.5, linestyle='--')
for bar, val in zip(bars, r2_vals):
    ax.text(val + (0.03 if val >= 0 else -0.10),
            bar.get_y() + bar.get_height()/2,
            f'{val:.2f}', va='center', fontsize=11, color='white')
ax.set_xlabel('$R^2$'); ax.set_title('$R^2$ interpretation')
ax.set_xlim(-0.6, 1.2); ax.grid(True, axis='x')

plt.tight_layout()
plt.savefig('img/m01_regression_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 2. Confusion Matrix

|  | Predicted Positive | Predicted Negative |
|--|---|---|
| **Actually Positive** | **TP** True Positive | **FN** False Negative (miss) |
| **Actually Negative** | **FP** False Positive (false alarm) | **TN** True Negative |

Every classification metric is derived from these four numbers.

- Medical diagnosis: a **FN** (missed cancer) is catastrophic — prioritise Recall
- Spam filter: a **FP** (blocking a legitimate email) is more costly — prioritise Precision
- Fraud detection: balance both — use F1

In [ ]:
np.random.seed(1)
X_clf, y_clf = make_classification(n_samples=500, n_features=10, n_informative=5, random_state=1)
X_tr, X_te, y_tr, y_te = train_test_split(X_clf, y_clf, test_size=0.3, random_state=1)
model   = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
y_pred  = model.predict(X_te)
y_prob  = model.predict_proba(X_te)[:, 1]
cm      = confusion_matrix(y_te, y_pred)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Confusion Matrix', fontsize=13)

ax = axes[0]
cmap_cm = LinearSegmentedColormap.from_list('cm', ['#1a1a1a', '#004d66'], N=100)
im = ax.imshow(cm, cmap=cmap_cm, vmin=0)
plt.colorbar(im, ax=ax)
cell_labels = [['TN','FP'],['FN','TP']]
cell_colors = [[RED,ORANGE],[ORANGE,GREEN]]
for i in range(2):
    for j in range(2):
        ax.text(j, i, f'{cell_labels[i][j]}\n{cm[i,j]}',
                ha='center', va='center', fontsize=14,
                color=cell_colors[i][j], fontweight='bold')
ax.set_xticks([0,1]); ax.set_yticks([0,1])
ax.set_xticklabels(['Pred Neg','Pred Pos'], fontsize=11)
ax.set_yticklabels(['Actual Neg','Actual Pos'], fontsize=11)
ax.set_title(f'Confusion Matrix  n={len(y_te)}')

ax = axes[1]
ax.axis('off')
table_data = [
    ['Use case',         'Costly FN',   'Costly FP',    'Balance both'],
    ['Cancer detection', 'Miss cancer', 'Extra test',   '—'],
    ['Spam filter',      'Spam passes', 'Block legit',  '—'],
    ['Fraud detection',  'Miss fraud',  'Investigate',  'F1 score'],
    ['Metric to use',    'Recall up',   'Precision up', 'F1 up'],
]
tbl = ax.table(cellText=table_data[1:], colLabels=table_data[0], loc='center', cellLoc='center')
tbl.auto_set_font_size(False); tbl.set_fontsize(10); tbl.scale(1.4, 2.2)
for (r, c), cell in tbl.get_celld().items():
    cell.set_facecolor('#1a1a1a' if r > 0 else '#2a2a2a')
    cell.set_edgecolor('#444')
    cell.set_text_props(color='#eee')
ax.set_title('Which error costs more depends on the problem', pad=40)

plt.tight_layout()
plt.savefig('img/m02_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 3. Accuracy, Precision, Recall, F1

**Accuracy**:
$$\text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN}$$
Useless on imbalanced datasets.

**Precision** — of all predicted positives, how many were actually positive:
$$\text{Precision} = \frac{TP}{TP + FP}$$

**Recall** — of all actual positives, how many did we catch:
$$\text{Recall} = \frac{TP}{TP + FN}$$

**F1** — harmonic mean of precision and recall:
$$F_1 = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}$$

Why harmonic mean? It is dominated by the **lower** value. Precision=1.0 and Recall=0.01 gives $F_1 \approx 0.02$, not $0.50$.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Accuracy, Precision, Recall, F1', fontsize=13)

ax = axes[0]
thresholds = np.linspace(0.01, 0.99, 200)
precisions, recalls, f1s = [], [], []
for thr in thresholds:
    yp = (y_prob >= thr).astype(int)
    if yp.sum() == 0:
        precisions.append(1.0); recalls.append(0.0); f1s.append(0.0); continue
    tn_, fp_, fn_, tp_ = confusion_matrix(y_te, yp).ravel()
    prec = tp_/(tp_+fp_) if (tp_+fp_) > 0 else 0
    rec  = tp_/(tp_+fn_) if (tp_+fn_) > 0 else 0
    f1   = 2*prec*rec/(prec+rec) if (prec+rec) > 0 else 0
    precisions.append(prec); recalls.append(rec); f1s.append(f1)
best_idx = np.argmax(f1s)
ax.plot(thresholds, precisions, color=ACCENT, lw=2.5, label='Precision')
ax.plot(thresholds, recalls,    color=ORANGE, lw=2.5, label='Recall')
ax.plot(thresholds, f1s,        color=GREEN,  lw=2.5, label='F1')
ax.axvline(thresholds[best_idx], color='white', lw=1.5, linestyle='--',
           label=f'Best F1 @ {thresholds[best_idx]:.2f}')
ax.set_xlabel('Threshold'); ax.set_ylabel('Score')
ax.set_title('Precision / Recall tradeoff vs threshold')
ax.legend(fontsize=9); ax.grid(True)

ax = axes[1]
prec_curve, rec_curve, _ = precision_recall_curve(y_te, y_prob)
pr_auc = auc(rec_curve, prec_curve)
ax.plot(rec_curve, prec_curve, color=PURPLE, lw=2.5, label=f'PR curve (AUC={pr_auc:.2f})')
ax.axhline(y_te.mean(), color='#555', lw=1.5, linestyle='--',
           label=f'Baseline (prevalence={y_te.mean():.2f})')
ax.scatter(recalls[best_idx], precisions[best_idx], color=GREEN, s=100, zorder=5, label='Best F1 point')
ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curve')
ax.legend(fontsize=9); ax.grid(True)
ax.set_xlim(0,1); ax.set_ylim(0, 1.05)

ax = axes[2]
p_arr = np.linspace(0.01, 0.99, 200); r_fix = 0.5
ax.plot(p_arr, (p_arr+r_fix)/2,              color=ORANGE, lw=2.5, linestyle='--', label='Arithmetic mean')
ax.plot(p_arr, 2*p_arr*r_fix/(p_arr+r_fix),  color=GREEN,  lw=2.5, label='F1 = Harmonic mean')
ax.axvline(r_fix, color='white', lw=1, linestyle=':', label=f'Recall fixed at {r_fix}')
ax.set_xlabel('Precision'); ax.set_ylabel('Score')
ax.set_title('F1 is dominated by the lower value')
ax.legend(fontsize=9); ax.grid(True)

plt.tight_layout()
plt.savefig('img/m03_precision_recall_f1.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4. Classification Threshold Effects

Most classifiers output a **probability score** $\hat{p} \in [0,1]$. The default threshold 0.5 is rarely optimal.

- **Lower threshold** → more positives predicted → Recall up, Precision down, more FP
- **Higher threshold** → fewer positives predicted → Precision up, Recall down, more FN

Adjusting the threshold is the cheapest lever after training. The right value depends on the **business cost** of each error type.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Classification Threshold Effects', fontsize=13)

ax = axes[0]
bins_sc = np.linspace(0, 1, 40)
ax.hist(y_prob[y_te==0], bins=bins_sc, alpha=0.6, color=ACCENT, density=True, label='True Negative')
ax.hist(y_prob[y_te==1], bins=bins_sc, alpha=0.6, color=ORANGE, density=True, label='True Positive')
for thr, color, ls in [(0.3, GREEN, '--'), (0.5, 'white', '-'), (0.7, RED, ':')]:
    ax.axvline(thr, color=color, lw=2, linestyle=ls, label=f'thr={thr}')
ax.set_xlabel('Predicted probability'); ax.set_ylabel('Density')
ax.set_title('Score distributions — overlap = error zone')
ax.legend(fontsize=8); ax.grid(True)

ax = axes[1]
thrs2 = np.linspace(0.01, 0.99, 200)
tp_l, fp_l, fn_l, tn_l = [], [], [], []
for thr in thrs2:
    yp = (y_prob >= thr).astype(int)
    tn_, fp_, fn_, tp_ = confusion_matrix(y_te, yp, labels=[0,1]).ravel()
    tp_l.append(tp_); fp_l.append(fp_); fn_l.append(fn_); tn_l.append(tn_)
ax.plot(thrs2, tp_l, color=GREEN,  lw=2, label='TP')
ax.plot(thrs2, tn_l, color=ACCENT, lw=2, label='TN')
ax.plot(thrs2, fp_l, color=ORANGE, lw=2, linestyle='--', label='FP')
ax.plot(thrs2, fn_l, color=RED,    lw=2, linestyle='--', label='FN')
ax.axvline(0.5, color='white', lw=1.5, linestyle=':', label='default thr=0.5')
ax.set_xlabel('Threshold'); ax.set_ylabel('Count')
ax.set_title('TP / TN / FP / FN vs threshold')
ax.legend(fontsize=9); ax.grid(True)

ax = axes[2]
np.random.seed(99)
n_total = 1000; y_imb = np.array([0]*950 + [1]*50)
preds_imb = {
    'Always 0':   np.zeros(n_total),
    'Random 50%': np.random.randint(0,2,n_total).astype(float),
    'Good model': np.where(y_imb==1, np.random.uniform(0.6,1.0,n_total),
                                     np.random.uniform(0.0,0.4,n_total)),
}
x_pos = np.arange(4); width = 0.25
for idx, (name, preds) in enumerate(preds_imb.items()):
    yp_hard = (preds >= 0.5).astype(int)
    tn_, fp_, fn_, tp_ = confusion_matrix(y_imb, yp_hard, labels=[0,1]).ravel()
    acc  = (tp_+tn_)/n_total
    prec = tp_/(tp_+fp_) if (tp_+fp_) > 0 else 0
    rec  = tp_/(tp_+fn_) if (tp_+fn_) > 0 else 0
    f1   = 2*prec*rec/(prec+rec) if (prec+rec) > 0 else 0
    ax.bar(x_pos+idx*width, [acc,prec,rec,f1], width,
           color=[RED,ORANGE,GREEN][idx], alpha=0.8, label=name)
ax.set_xticks(x_pos+width); ax.set_xticklabels(['Accuracy','Precision','Recall','F1'])
ax.set_ylim(0, 1.15); ax.set_ylabel('Score')
ax.set_title('Imbalanced (95/5%) — accuracy is misleading')
ax.legend(fontsize=9); ax.grid(True, axis='y')

plt.tight_layout()
plt.savefig('img/m04_threshold.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5. ROC Curve & AUC

The **ROC curve** plots TPR vs FPR as the threshold sweeps from 1 to 0:
$$\text{TPR} = \frac{TP}{TP+FN} \qquad \text{FPR} = \frac{FP}{FP+TN}$$

**AUC**: $1.0$ = perfect, $0.5$ = random, $<0.5$ = worse than random.

**Probabilistic interpretation**:
$$\text{AUC} = P(\hat{p}_{\text{pos}} > \hat{p}_{\text{neg}})$$

**ROC vs PR curve**: on heavily imbalanced datasets ROC can be overly optimistic. Prefer the **PR curve** when positives are rare.

In [ ]:
np.random.seed(2)
X2, y2 = make_classification(n_samples=600, n_features=12, n_informative=6, random_state=2)
X_tr2, X_te2, y_tr2, y_te2 = train_test_split(X2, y2, test_size=0.3, random_state=2)
models_roc = {
    'Logistic Regression': LogisticRegression(max_iter=1000).fit(X_tr2, y_tr2),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=2).fit(X_tr2, y_tr2),
    'Random baseline':     DummyClassifier(strategy='uniform', random_state=2).fit(X_tr2, y_tr2),
}
colors_roc = [ACCENT, GREEN, RED]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('ROC Curve & AUC', fontsize=13)

ax = axes[0]
rf_fpr, rf_tpr = None, None
for (name, m), color in zip(models_roc.items(), colors_roc):
    prob = m.predict_proba(X_te2)[:,1]
    fpr, tpr, _ = roc_curve(y_te2, prob)
    roc_auc = auc(fpr, tpr)
    ls = '--' if 'baseline' in name else '-'
    ax.plot(fpr, tpr, color=color, lw=2.5, linestyle=ls, label=f'{name} (AUC={roc_auc:.3f})')
    if name == 'Random Forest': rf_fpr, rf_tpr = fpr, tpr
ax.plot([0,1],[0,1], color='#555', lw=1.5, linestyle=':')
if rf_fpr is not None: ax.fill_between(rf_fpr, rf_tpr, alpha=0.08, color=GREEN)
ax.set_xlabel('FPR'); ax.set_ylabel('TPR (Recall)')
ax.set_title('ROC Curve — top-left corner = perfect')
ax.legend(fontsize=9); ax.grid(True)
ax.set_xlim(0,1); ax.set_ylim(0,1.02)

ax = axes[1]
rf_prob = models_roc['Random Forest'].predict_proba(X_te2)[:,1]
bins_auc = np.linspace(0, 1, 40)
ax.hist(rf_prob[y_te2==0], bins=bins_auc, alpha=0.6, color=ACCENT, density=True, label='True Negatives')
ax.hist(rf_prob[y_te2==1], bins=bins_auc, alpha=0.6, color=ORANGE, density=True, label='True Positives')
fpr_rf, tpr_rf, _ = roc_curve(y_te2, rf_prob)
roc_auc_rf = auc(fpr_rf, tpr_rf)
ax.set_xlabel('Predicted probability'); ax.set_ylabel('Density')
ax.set_title(f'AUC = P(score_pos > score_neg) = {roc_auc_rf:.3f}')
ax.legend(fontsize=10); ax.grid(True)

plt.tight_layout()
plt.savefig('img/m05_roc_auc.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 6. Log-loss & Cross-entropy

**Log-loss** measures the quality of **probability estimates**, not just hard labels:
$$\mathcal{L}_{\text{log}} = -\frac{1}{n}\sum_{i=1}^n \left[ y_i \log(\hat{p}_i) + (1-y_i)\log(1-\hat{p}_i) \right]$$

Confident wrong predictions are penalised exponentially: predicting $\hat{p}=0.99$ for the wrong class costs $-\log(0.01) \approx 4.6$.

**Cross-entropy** — multi-class generalisation:
$$\mathcal{L}_{\text{CE}} = -\frac{1}{n}\sum_{i=1}^n \sum_{c=1}^C y_{ic} \log(\hat{p}_{ic})$$

**Information theory link**: minimising cross-entropy = minimising KL divergence between the true and predicted distributions:
$$H(p, q) = H(p) + D_{KL}(p \| q)$$

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Log-loss & Cross-entropy', fontsize=13)

ax = axes[0]
p_range = np.linspace(1e-6, 1-1e-6, 400)
ax.plot(p_range, -np.log(p_range),   color=GREEN,  lw=2.5, label='$y=1$:  $-\\log(\\hat{p})$')
ax.plot(p_range, -np.log(1-p_range), color=ORANGE, lw=2.5, label='$y=0$:  $-\\log(1-\\hat{p})$')
ax.axvline(0.5, color='#555', lw=1, linestyle=':')
ax.set_ylim(0, 5)
ax.set_xlabel('$\\hat{p}$'); ax.set_ylabel('Loss')
ax.set_title('Confident wrong predictions penalised heavily')
ax.legend(fontsize=10); ax.grid(True)

ax = axes[1]
scenarios = {
    'Confident correct (y=1, p=0.95)': (1, 0.95),
    'Uncertain correct (y=1, p=0.55)': (1, 0.55),
    'Uncertain wrong   (y=1, p=0.45)': (1, 0.45),
    'Confident wrong   (y=1, p=0.05)': (1, 0.05),
}
losses_s = [-(y*np.log(p)+(1-y)*np.log(1-p)) for y, p in scenarios.values()]
colors_s  = [GREEN, ACCENT, ORANGE, RED]
bars = ax.barh(list(scenarios.keys()), losses_s, color=colors_s, alpha=0.8, height=0.5)
for bar, val in zip(bars, losses_s):
    ax.text(val+0.05, bar.get_y()+bar.get_height()/2,
            f'{val:.2f}', va='center', fontsize=11, color='white')
ax.set_xlabel('Log-loss'); ax.set_title('Penalty by prediction confidence')
ax.grid(True, axis='x')

ax = axes[2]
np.random.seed(10)
n_cal = 300; y_cal = np.random.randint(0, 2, n_cal)
p_calib  = np.clip(y_cal + np.random.normal(0, 0.25, n_cal), 0.01, 0.99)
p_overc  = np.clip(np.where(y_cal==1, np.random.uniform(0.8,1.0,n_cal), np.random.uniform(0.0,0.2,n_cal)), 0.01, 0.99)
p_underc = np.clip(np.where(y_cal==1, np.random.uniform(0.5,0.65,n_cal), np.random.uniform(0.35,0.5,n_cal)), 0.01, 0.99)
models_cal = [('Calibrated',p_calib,ACCENT),('Overconfident',p_overc,RED),('Underconfident',p_underc,ORANGE)]
logloss_vals = [log_loss(y_cal, m[1]) for m in models_cal]
auc_vals     = [auc(*roc_curve(y_cal, m[1])[:2]) for m in models_cal]
x_bar = np.arange(3); w = 0.35
ax.bar(x_bar-w/2, logloss_vals, w, color=[m[2] for m in models_cal], alpha=0.85, label='Log-loss (lower=better)')
ax.bar(x_bar+w/2, auc_vals,     w, color=[m[2] for m in models_cal], alpha=0.4,  label='AUC (higher=better)', hatch='//')
ax.set_xticks(x_bar); ax.set_xticklabels([m[0] for m in models_cal], fontsize=9)
ax.set_ylabel('Score'); ax.set_title('Log-loss measures calibration\nAUC measures ranking')
ax.legend(fontsize=9); ax.grid(True, axis='y')

plt.tight_layout()
plt.savefig('img/m06_logloss.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Summary Table

### Regression

| Metric | Formula | Notes |
|--------|---------|-------|
| MAE | $\\frac{1}{n}\\sum|y_i-\\hat{y}_i|$ | Robust to outliers — predicts median |
| MSE | $\\frac{1}{n}\\sum(y_i-\\hat{y}_i)^2$ | Penalises large errors — predicts mean |
| RMSE | $\\sqrt{\\text{MSE}}$ | Same unit as $y$, sensitive to outliers |
| R² | $1 - SS_{res}/SS_{tot}$ | Fraction of variance explained |

### Classification

| Metric | Formula | When to use |
|--------|---------|-------------|
| Accuracy | $(TP+TN)/n$ | Balanced classes only |
| Precision | $TP/(TP+FP)$ | Cost of false alarms is high |
| Recall | $TP/(TP+FN)$ | Cost of missing positives is high |
| F1 | $2PR/(P+R)$ | Imbalanced classes |
| AUC-ROC | Area under TPR vs FPR | Threshold-free ranking quality |
| Log-loss | $-\\frac{1}{n}\\sum y\\log\\hat{p}+(1-y)\\log(1-\\hat{p})$ | Quality of probability estimates |

---
*Back to: [`00b_statistical_indicators.ipynb`](./00b_statistical_indicators.ipynb)*